
## NHS Cancer Waiting Times — Gold: Trust × Month (the agent's table)

**Notebook 06 of the pipeline** — Gold layer, the cut my public-facing agent reads.

My three article cuts (03–05) each collapse the data one way: the trust ranking averages
every trust over 18 months, the national trend keeps every month but only for England.
My agent needs *both* axes at once — for one trust, its latest month **and** its full
18-month trend — so neither of those tables can serve it.

This notebook builds the missing cut: **one row per trust per month**, on the combined
62-day standard, with the England aggregate kept in so the agent can compare against it.
It is the single artifact the agent queries — and every number in it is computed here,
in my pipeline, never by the AI.

**Input:** `silver_62d.parquet`  
**Output:** `gold_62d_trust_month.parquet` (+ a human-readable CSV in `outputs/`)

In [4]:
# I check where this notebook is actually running from, and whether my helper is here.
import os
print("I am running in:", os.getcwd())
print("Python files I can see:", [f for f in os.listdir() if f.endswith(".py")])

I am running in: /Users/yusufismail/Documents/cancer_waiting_time/notebooks
Python files I can see: ['cwt_narrate.py', 'cwt_utils.py', 'cwt_query.py', 'cwt_answer.py']


In [5]:
# I show what's actually in my current folder, then locate my helper precisely.
from pathlib import Path

here = Path.cwd()
print("I am here:", here, "\n")

print("What's in this folder:")
for p in sorted(here.iterdir()):
    print("   ", p.name + ("/" if p.is_dir() else ""))

print("\ncwt_utils.py is actually at:")
for f in here.rglob("cwt_utils.py"):
    print("   ", f)

I am here: /Users/yusufismail/Documents/cancer_waiting_time/notebooks 

What's in this folder:
    .ipynb_checkpoints/
    00_data_exploration.ipynb
    01_bronze_ingestion.ipynb
    02_silver_transformation.ipynb
    03_gold_trust_lottery.ipynb
    04_gold_cancer_type.ipynb
    05_gold_routes.ipynb
    06_gold_trust_month.ipynb
    __pycache__/
    cwt_answer.py
    cwt_narrate.py
    cwt_query.py
    cwt_utils.py

cwt_utils.py is actually at:
    /Users/yusufismail/Documents/cancer_waiting_time/notebooks/cwt_utils.py


In [6]:
# I import my shared helpers so this notebook uses exactly the same loading, paths and
# rules as the rest of my pipeline. If it imports clean, I am consistent with 00–05.
import pandas as pd
import numpy as np
import cwt_utils as cwt

# I load my clean 62-day Silver file. Silver already did the heavy lifting: casing
# normalised, compliance recalculated from source counts, nulls preserved.
df62 = cwt.load_silver("62d")

print("62D Silver loaded:", df62.shape)
print("Months:", df62["period_month"].nunique(),
      f"({df62['period_month'].min()} to {df62['period_month'].max()})")

62D Silver loaded: (135359, 20)
Months: 18 (2023-10 to 2025-03)


## Step 1 — Filter to the combined 62-day standard

The 62-day standard has several referral routes (urgent suspected cancer, screening,
breast symptomatic, consultant upgrade). For my articles I told the *urgent* route's
story specifically. But the agent claims to report performance against **the 62-day
standard**, and the figure the 85% target officially attaches to is the **combined** one
across all routes — which lives in the data as `referral_route_clean == 'ALL ROUTES'`.

Using the combined figure is both more honest (it is what the target measures) and wider
in coverage: ALL ROUTES reports for 174 org codes versus 160 on the urgent route alone,
so the agent can answer for ~14 more trusts.

I fix three dimensions and, unlike my ranking notebook, I **keep** the England `Total`
row — the agent needs it for comparison, and keeping it here makes that a filter, never a join.

In [8]:
# I fix all three dimensions to the COMBINED 62-day standard — the figure the 85% target
# actually measures, and the one a member of the public means by "the 62-day standard":
#   Cancer_Type          == 'ALL CANCERS'    -> not a single tumour group (v1 scope: no splits)
#   referral_route_clean == 'ALL ROUTES'     -> the combined standard, NOT urgent-route only
#   Treatment_Modality   == 'ALL MODALITIES' -> surgery + drugs + radiotherapy combined
#
# I deliberately keep the England 'Total' row this time (I do NOT call exclude_total_row).
# The agent compares each trust to the England figure, and that figure lives right here as
# Org_Code == 'Total'. Keeping it makes the comparison a filter, never a join to a 2nd table.
agent = df62[
    (df62["Cancer_Type"] == "ALL CANCERS") &
    (df62["referral_route_clean"] == "ALL ROUTES") &
    (df62["Treatment_Modality"] == "ALL MODALITIES")
].copy()

print(f"Rows on the combined-standard basis: {len(agent):,}")
print(f"Distinct org codes (incl. England Total): {agent['Org_Code'].nunique()}")
print(f"Months: {agent['period_month'].nunique()}")
print(f"England Total row present: {(agent['Org_Code'] == 'Total').any()}")

Rows on the combined-standard basis: 2,659
Distinct org codes (incl. England Total): 174
Months: 18
England Total row present: True


## Step 2 — Verify the grain

With those three dimensions fixed, every remaining row should be a unique trust × month.
I prove it. If two rows shared a trust and a month, the agent would double-count that
trust's patients — so I assert uniqueness loudly here rather than let a silent bug reach
a public answer. This is the same discipline as my Silver join-key test.

In [10]:
# I prove the grain is exactly one row per trust per month. If this fails, something in the
# basis filter is letting a second dimension value through, and the agent would silently
# double-count — so I catch it here, not in production.
grain = agent.groupby(["Org_Code", "period_month"]).size()
n_dupes = int((grain > 1).sum())
print(f"Org-month combinations: {len(grain):,}")
print(f"Any with more than one row: {n_dupes}")
assert grain.max() == 1, f"Grain is not unique: {n_dupes} org-months have duplicate rows"
print("Grain verified: exactly one row per trust per month.")

Org-month combinations: 2,659
Any with more than one row: 0
Grain verified: exactly one row per trust per month.


## Step 3 — Shape the agent's table

I keep only the columns the agent needs and rename them to clean, stable names, so the
agent code never has to know Silver's naming. I carry `compliance_rate` straight through —
Silver already computed it from source counts, and **my code owns that maths once**; here
I only read it, I never recompute it.

I add `metric` as a *column* (value `'62d_combined'`), not a hardcoded assumption. When I
add the Faster Diagnosis or 31-day standards later they slot in as new rows under a new
metric label — no restructuring, no second table.

I also reconcile the counts: every patient is either within or after 62 days, so
`within_62d + breached_62d` must equal `total_patients`. I assert it.

In [12]:
# I select only what the agent needs and rename to clean, stable names, decoupling the
# agent from Silver's naming. Mapping (Silver -> agent table):
#   Org_Code -> org_code, Org_Name -> org_name, Total -> total_patients,
#   Within -> within_62d, After -> breached_62d. compliance_rate carries as-is.
gold = agent[[
    "period_month", "period_date",
    "Org_Code", "Org_Name",
    "Total", "Within", "After", "compliance_rate",
]].rename(columns={
    "Org_Code": "org_code",
    "Org_Name": "org_name",
    "Total": "total_patients",
    "Within": "within_62d",
    "After": "breached_62d",
}).copy()

# I add the metric as a COLUMN, not a hardcoded value, so future standards are new rows.
gold.insert(0, "metric", "62d_combined")

# I verify the counts reconcile: total must equal within + breached for every row (allowing
# for NHS's .5 shared-pathway half-patients and floating point). I fail loudly otherwise.
recomputed = gold["within_62d"] + gold["breached_62d"]
max_gap = (gold["total_patients"] - recomputed).abs().max()
print(f"Max reconciliation gap (within + breached vs total): {max_gap:.2e}")
assert max_gap < 1e-6, "Counts do not reconcile — within + breached != total for some row"
print("Counts reconcile: within_62d + breached_62d == total_patients for every row.")

print("\nAgent table schema:")
print(gold.dtypes.to_string())

Max reconciliation gap (within + breached vs total): 0.00e+00
Counts reconcile: within_62d + breached_62d == total_patients for every row.

Agent table schema:
metric                     object
period_month               object
period_date        datetime64[ns]
org_code                   object
org_name                   object
total_patients            float64
within_62d                float64
breached_62d              float64
compliance_rate           float64


## Step 4 — Write the Gold table

I persist both formats, exactly as my other Gold notebooks do: Parquet for the pipeline
(typed, the artifact the agent loads at startup), and a CSV in `outputs/` so the table is
readable on GitHub and I can eyeball it without opening Python.

In [14]:
# I sort for a clean, readable file: by trust, then chronologically within each trust.
gold = gold.sort_values(["org_code", "period_date"]).reset_index(drop=True)

# I write both formats, as I do for every Gold table.
parquet_path = cwt.GOLD_DIR / "gold_62d_trust_month.parquet"
csv_path     = cwt.OUTPUTS_DIR / "gold_62d_trust_month.csv"

gold.to_parquet(parquet_path, index=False)
gold.to_csv(csv_path, index=False)

print("Gold table written — the single artifact the agent reads:")
print(f"  {parquet_path.name}  ({len(gold):,} rows × {gold.shape[1]} cols)")
print(f"  {csv_path.name}  (same data, human-readable)")

Gold table written — the single artifact the agent reads:
  gold_62d_trust_month.parquet  (2,659 rows × 9 cols)
  gold_62d_trust_month.csv  (same data, human-readable)


## Step 5 — Verify, and prove the trend is now there

I reload the Parquet as if I were the agent, confirm the shape, and run the spot-check that
**neither old Gold table could answer**: one trust, every month, compliance intact. I pick
Shrewsbury and Telford (RXW) — the trust from my own worked example — and show the matching
England figure for the latest month, which is the exact comparison the agent makes.

In [16]:
# I reload the Parquet as if I were the agent starting fresh, and confirm the shape.
check = pd.read_parquet(parquet_path)
print(f"Reloaded: {check.shape[0]:,} rows × {check.shape[1]} cols")
print(f"Distinct trusts incl. England Total: {check['org_code'].nunique()}")
print(f"Months: {check['period_month'].nunique()} "
      f"({check['period_month'].min()} to {check['period_month'].max()})")
print(f"England Total row present: {(check['org_code'] == 'Total').any()}")

# I run the spot-check that was IMPOSSIBLE with either old Gold table: one trust, every
# month, with its compliance trend intact. This is the whole reason the new cut exists.
rxw = (check[check["org_code"] == "RXW"]
       .sort_values("period_month")
       [["period_month", "total_patients", "within_62d", "breached_62d", "compliance_rate"]])
print(f"\nSpot-check — Shrewsbury and Telford (RXW), all {len(rxw)} months:")
print(rxw.to_string(index=False))

# I show the matching England figure for the latest month — the comparison the agent makes.
latest = check["period_month"].max()
eng = check[(check["org_code"] == "Total") & (check["period_month"] == latest)]
print(f"\nEngland combined, {latest}: {eng['compliance_rate'].iloc[0]*100:.1f}%")
print("(the agent compares each trust to this, and to the 85% standard)")

Reloaded: 2,659 rows × 9 cols
Distinct trusts incl. England Total: 174
Months: 18 (2023-10 to 2025-03)
England Total row present: True

Spot-check — Shrewsbury and Telford (RXW), all 18 months:
period_month  total_patients  within_62d  breached_62d  compliance_rate
     2023-10           338.0       190.5         147.5         0.563609
     2023-11           309.5       144.5         165.0         0.466882
     2023-12           269.0       142.0         127.0         0.527881
     2024-01           352.0       176.5         175.5         0.501420
     2024-02           313.5       171.0         142.5         0.545455
     2024-03           333.0       193.0         140.0         0.579580
     2024-04           322.0       192.5         129.5         0.597826
     2024-05           268.0       166.5         101.5         0.621269
     2024-06           279.0       160.0         119.0         0.573477
     2024-07           310.5       166.5         144.0         0.536232
     2024-08  

In [17]:
# The contract every query returns — one frozen shape. (Sketch for you to react to.)
result = {
    "trust":   {"org_code": "RXW",
                "org_name": "The Shrewsbury and Telford Hospital NHS Trust"},
    "metric":  "62d_combined",
    "latest":  {"period_month": "2025-03", "compliance_rate": 0.6655,
                "total_patients": 288.5, "within_62d": 192.0, "breached_62d": 96.5},
    "trend":   [{"period_month": "2023-10", "compliance_rate": 0.5636}, ...],  # all 18
    "england": {"period_month": "2025-03", "compliance_rate": 0.7140},
    "standard": 0.85,
    "provenance": {"data_month": "2025-03",
                   "source_url": "https://www.england.nhs.uk/statistics/.../cancer-waiting-times/"},
    "caveats": [],
}

In [18]:
# I smoke-test the query layer against my real Gold table.
import cwt_query as q

df = q.load_gold_table(cwt.GOLD_DIR / "gold_62d_trust_month.parquet")

print("resolve 'shrewsbury':", q.resolve_trust(df, "shrewsbury"))
res = q.get_trust_performance(df, "RXW")
print("trend points:", len(res.trend), "| latest:", res.latest.period_month,
      f"{res.latest.compliance_rate*100:.1f}%",
      "| England:", f"{res.england.compliance_rate*100:.1f}%")
print("caveats:", res.caveats)

resolve 'shrewsbury': [{'org_code': 'RXW', 'org_name': 'THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST'}]
trend points: 18 | latest: 2025-03 66.6% | England: 71.4%
caveats: []


In [1]:
# I enter my key into a hidden prompt, then strip any invisible whitespace — including the
# non-breaking space (Option+Space on a Mac) that broke the last attempt before it sent.
import os, getpass, re
os.environ["ANTHROPIC_API_KEY"] = re.sub(r"\s+", "", getpass.getpass("Anthropic API key (hidden): "))

import cwt_utils as cwt, cwt_query as q, cwt_answer as a, cwt_narrate as n
df  = q.load_gold_table(cwt.GOLD_DIR / "gold_62d_trust_month.parquet")
res = q.get_trust_performance(df, "RXW")
f   = a.build_findings(res)
print(n.narrate(f, "how is Shrewsbury performing?"))
print()
print(n.narrate(f, "how long will I wait at Shrewsbury?"))

Anthropic API key (hidden):  ········


In March 2025, about two in three patients at Shrewsbury and Telford Hospital NHS Trust began their cancer treatment within 62 days of an urgent referral for suspected cancer (66.6%). This is slightly below the England-wide figure of 71.4% for that month, and below the 85% standard the NHS aims for. The trust has improved steadily — this is its best performance in the past 18 months, up from 56.4% in October 2023.

Source: NHS England Cancer Waiting Times, data to March 2025.

No one can predict how long you personally will wait — it depends on your individual circumstances. What I can tell you is that at Shrewsbury and Telford Hospital NHS Trust in March 2025, about two in three patients (66.6%) began their cancer treatment within 62 days of an urgent referral for suspected cancer. That's slightly below the England-wide figure of 71.4% for that month.

Source: NHS England Cancer Waiting Times, data to March 2025.


In [27]:
import os, getpass, re
os.environ["ANTHROPIC_API_KEY"] = re.sub(r"\s+", "", getpass.getpass("Anthropic API key (hidden): "))


import cwt_batch; cwt_batch.regenerate(['NTV'])

Anthropic API key (hidden):  ········


[1/1] regenerated NTV  Csh Surrey

Updated 1 trusts in /Users/yusufismail/Documents/cancer_waiting_time/data/gold/answers_62d.json
